# Tutorial 2: Hybrid Search

*Level: Intermediate*

---

Dense retrieval excels at capturing semantic similarity but struggles with lexical precision. When a query contains specific scientific terms, a dense model may retrieve conceptually related documents that don't contain the exact terminology, missing the most directly relevant results. Sparse retrieval addresses this through keyword matching: it scores documents based on exact term overlap, making it naturally strong where dense models fall short.

## What we'll build

In this tutorial we combine dense and sparse retrieval into a **hybrid search** pipeline, leveraging both semantic understanding and exact matching to improve retrieval performance.

We compare three configurations:

| Configuration | What it uses |
|---|---|
| Dense only | bge-base |
| Sparse only | BM25 keyword matching |
| Dense + Sparse (RRF) | Both fused with Reciprocal Rank Fusion |

We will also conduct a small **RRF tuning experiment** to illustrate how the balance between dense and sparse signals can be adjusted to squeeze out additional retrieval quality.

## The dataset: SciFact

We use **SciFact**, a scientific fact-checking dataset from [BEIR](https://huggingface.co/datasets/BeIR/scifact) (Benchmarking Information Retrieval), a standard evaluation suite for Information Retrieval systems. It assesses retrieval models by matching scientific claims to supporting or refuting evidence in biomedical texts.

What makes it suitable for our tutorial: it comes with **ground-truth relevance judgments** (qrels) that tell us, for each query, which documents are relevant. This is what allows us to measure retrieval quality objectively rather than eyeballing results.

To keep ingestion fast, we will only ingest the documents that appear in the qrels test split: the subset of the corpus that is referenced by test queries.

## What we'll evaluate

For each configuration we track how well the retrieved documents match the query, measured with standard IR metrics (NDCG@K, MRR, Recall@K, Precision@K).

## What we'll use

- **FastEmbed:** Qdrant's lightweight embedding library. It runs ONNX-optimized models and supports GPU acceleration.
- **Qdrant:** our vector database and search engine. It stores the document embeddings and handles similarity search.
- **ranx:** a fast ranking evaluation library for retrieval metrics computation against ground truth.

> **Note:** This notebook runs on CPU. Ingestion takes a few minutes. For faster iteration or experimenting with the full SciFact corpus, switch to a T4 GPU runtime in Colab.
---

## 0. Setup

We start by installing the required libraries:

- **fastembed**: Qdrant's lightweight embedding library.
- **qdrant-client**: the Python client for Qdrant. It allows you to interact with your Qdrant cluster directly from Python.
- **datasets**: Hugging Face's library for loading datasets.
- **ranx**: a ranking evaluation library to compute retrieval metrics against ground truth.
- **tqdm**: progress bars for long-running loops.

In [ ]:
# GPU optional: if you switched to a GPU runtime, replace the fastembed install below
# with these two lines instead:
# !pip install onnxruntime-gpu -i https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ -qqq
# !pip install fastembed-gpu qdrant-client datasets ranx tqdm -qqq

!pip install fastembed qdrant-client datasets ranx tqdm -qqq

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.4/467.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import time
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams,
    PointStruct, SparseVector,
    Prefetch, FusionQuery, Fusion,
    RrfQuery, Rrf,
)
from fastembed import TextEmbedding, SparseTextEmbedding
from ranx import Qrels, Run, evaluate

## 1. Create a Qdrant Cluster and setup a client connection

If you do not already have a Qdrant cluster, follow these steps to create one:

- Register for a [Qdrant Cloud account](https://cloud.qdrant.io/) using your email, Google, or Github credentials.
- Under Create a Free Cluster, enter a cluster name and select your preferred cloud provider and region.
- Click Create Free Cluster.
- Copy the API key when prompted and store it somewhere safe as it won't be displayed again.
- Copy the Cluster Endpoint. It should look something like `https://xxx.cloud.qdrant.io`.


Create a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io).  
In Colab: open **Secrets** (key icon on the left sidebar) and add:
- `QDRANT_URL` : your cluster endpoint e.g. `https://xyz.us-east4-0.gcp.cloud.qdrant.io`
- `QDRANT_API_KEY` : your API key

Next create a client connection to your Qdrant cluster

In [ ]:
from google.colab import userdata

QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("Connected. Collections:", [c.name for c in client.get_collections().collections])

Connected. Collections: ['tutorial1_scifact-collection', 'tutorial2_scifact']


---
## 2. Load SciFact

SciFact is fully public on Hugging Face. We load three splits:
- **corpus**: the biomedical texts we'll index
- **queries**: short scientific claims
- **qrels**: ground-truth relevance judgments

We concatenate `title + text` for each document. Titles in scientific abstracts are typically informative and keyword-rich, so including them alongside the abstract generally improves retrieval quality.

To keep ingestion fast, we will only ingest the documents that appear in the qrels.

### Relevance judgments (qrels)

Qrels (query relevance judgments) are the ground truth that makes evaluation possible. They map each query to its relevant documents with a relevance score of **1**. Any (query, document) pair absent from the qrels is treated as irrelevant by the evaluation framework.

We load the **test split**, which contains 300 queries.

In [ ]:
corpus_dataset = load_dataset("BeIR/scifact", "corpus", split="corpus")

doc_ids      = [doc["_id"]   for doc in corpus_dataset]
doc_titles   = [doc["title"] for doc in corpus_dataset]
doc_texts    = [doc["text"]  for doc in corpus_dataset]
doc_passages = [(t + ". " + x).strip() for t, x in zip(doc_titles, doc_texts)]

print(f"Corpus: {len(doc_ids)} documents")

queries_dataset = load_dataset("BeIR/scifact", "queries", split="queries")
queries = {q["_id"]: q["text"] for q in queries_dataset}

qrels_dataset = load_dataset("BeIR/scifact-qrels", split="test")
qrels_dict    = defaultdict(dict)
for row in qrels_dataset:
    qrels_dict[str(row["query-id"])][str(row["corpus-id"])] = row["score"]

eval_queries = {qid: queries[qid] for qid in qrels_dict if qid in queries}
qrels_ranx   = Qrels(dict(qrels_dict))

# Filter corpus to only documents referenced in qrels
# Ranx treats any document not in qrels for a given query as grade 0
# so evaluation remains fully valid with this filtered corpus
qrel_doc_ids = set(
    doc_id
    for doc_dict in qrels_dict.values()
    for doc_id in doc_dict.keys()
)

mask         = [did in qrel_doc_ids for did in doc_ids]
doc_ids      = [x for x, m in zip(doc_ids,      mask) if m]
doc_titles   = [x for x, m in zip(doc_titles,   mask) if m]
doc_texts    = [x for x, m in zip(doc_texts,    mask) if m]
doc_passages = [x for x, m in zip(doc_passages, mask) if m]

# Stats
counts = [len(v) for v in qrels_dict.values()]
print(f"Filtered corpus : {len(doc_ids)} documents")
print(f"Evaluation queries : {len(eval_queries)}")
print(f"Relevant docs per query: mean={np.mean(counts):.1f}  min={min(counts)}  max={max(counts)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet:   0%|          | 0.00/4.47M [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

Corpus: 5183 documents


...

train.tsv:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/919 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]

Filtered corpus : 283 documents
Evaluation queries : 300
Relevant docs per query: mean=1.1  min=1  max=5


## 3. Load the models

We load:

- `bge-base-en-v1.5` a dense embedding model for semantic retrieval.
- `BM25` a lexical model for keyword-based sparse retrieval.

### Dense vectors

Dense vectors are fixed-size numerical representations that capture the semantic meaning of data through similarity in vector space. In Qdrant, they are indexed using HNSW (Hierarchical Navigable Small World), which enables fast approximate nearest neighbor search. We use bge-base-en-v1.5 as our embedding model to produce these vectors.

> **Note:** `bge-base-en-v1.5` has a context window of 512 tokens. In this tutorial we don't chunk documents and tolerate any truncation that occurs. In production, verify that your document lengths fit within your model's context window.

### Sparse vectors

Sparse vectors are high-dimensional vectors filled mostly with zeros. Only a few dimensions are non-zero, each typically corresponding to a token in the vocabulary. Because most values are zero, they are stored efficiently in Qdrant using an **inverted index**: only the non-zero indices and their values are stored. This is also why, unlike dense vectors, we do not specify a dimension when defining a sparse vector.

#### BM25

BM25 (Best Matching 25) is one of the most widely used sparse retrieval models. It builds on TF-IDF (Term Frequency–Inverse Document Frequency), which measures how important a term is based on two ideas:

* Term frequency (TF): how often a word appears in a document.
* Inverse document frequency (IDF): how rare that word is across all documents.

Together, they highlight terms that are both **frequent** in a document and **distinctive** in the corpus.

BM25 improves on this by making the scoring more realistic in two ways:

* It reduces the impact of repeated words, so that seeing a term many times does not endlessly increase its importance.
* It adjusts for document length, so that longer documents are not unfairly favored.

When using BM25 in Qdrant, you only need to provide sparse term representations. Qdrant computes and maintains IDF server-side, and applies it automatically during scoring at query time.

You can read more about keyword search with sparse vectors [here](https://qdrant.tech/course/essentials/day-3/sparse-retrieval-demo/).

In [ ]:
DENSE_MODEL_NAME   = "BAAI/bge-base-en-v1.5"
DENSE_DIM          = 768
SPARSE_MODEL_NAME  = "Qdrant/bm25"


In [ ]:
print("Loading dense model...")
# dense_model = TextEmbedding(model_name=DENSE_MODEL_NAME, providers=["CUDAExecutionProvider"]) #If you're using GPU
dense_model = TextEmbedding(model_name=DENSE_MODEL_NAME)

print("Loading sparse model...")
sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL_NAME)

print("All models ready.")

Loading dense model...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading sparse model...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

All models ready.


---
## 4. Create the hybrid collection and ingest

We create our collection with two **named vectors**:

- **dense**: 768-dim vectors from `bge-base-en-v1.5` used for semantic retrieval.
- **sparse**: sparse vectors from `BM25` used for keyword matching.

Each point looks like this:

```python
PointStruct(
    id=0,
    vector={
        "dense":  [...],         # 768-dim float vector
        "sparse": SparseVector(
            indices=[23, 401, 1205, ...],  # non-zero token indices
            values= [0.4, 0.7, 0.2, ...]  # corresponding BM25 weights
        ),
    },
    payload={
        "doc_id": "...",
        "title":  "...",
        "text":   "..."
    }
)
```




In [ ]:
COLLECTION_NAME    = "tutorial2_scifact"

In [ ]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing '{COLLECTION_NAME}'")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    },
)
print(f"Collection '{COLLECTION_NAME}' created.")

Deleted existing 'tutorial2_scifact'
Collection 'tutorial2_scifact' created.


Define the data ingestion function: We embed each passage with both models and batch upload our points.

Resume logic: if the session is interrupted, it picks up from the last uploaded point.

In [ ]:
def ingest(doc_ids, doc_titles, doc_texts, doc_passages, batch_size=32):
    total     = len(doc_ids)
    start_idx = client.count(COLLECTION_NAME).count

    if start_idx >= total:
        print(f"Already ingested {total} points")
        return

    if start_idx > 0:
        print(f"Resuming from index {start_idx}")

    t0 = time.time()
    for i in tqdm(range(start_idx, total, batch_size), desc="Ingesting"):
        end      = min(i + batch_size, total)
        passages = doc_passages[i:end]

        dense_vecs  = [e.tolist() for e in dense_model.embed(passages)]
        sparse_embs = list(sparse_model.embed(passages))

        client.upsert(
            collection_name=COLLECTION_NAME,
            points=[
                PointStruct(
                    id=i + j,
                    vector={
                        "dense": dense_vecs[j],
                        "sparse": SparseVector(
                            indices=sparse_embs[j].indices.tolist(),
                            values=sparse_embs[j].values.tolist(),
                        ),
                    },
                    payload={
                        "doc_id": doc_ids[i + j],
                        "title":  doc_titles[i + j],
                        "text":   doc_texts[i + j],
                    },
                )
                for j in range(end - i)
            ], 
            wait=True,
        )

    print(f"Done in {time.time()-t0:.1f}s. {client.count(COLLECTION_NAME).count} points ingested.")

In [ ]:
ingest(doc_ids, doc_titles, doc_texts, doc_passages, batch_size=64)

Ingesting: 100%|██████████| 5/5 [06:13<00:00, 74.75s/it]

Done in 373.8s. 283 points ingested.


---
## 5. Define search configurations and evaluate

### Universal Query API

All three configurations use Qdrant's Universal Query API, which allows retrieval, fusion, and reranking to be executed server-side in a single `query_points` call.

- For dense-only and sparse-only search, the process is straightforward: a single query vector is used, and results are returned directly.
- For hybrid search, we combine multiple retrieval signals within the same request.

### Hybrid search with prefetch

Hybrid search is implemented using `prefetch`, which runs multiple retrieval passes in parallel before final ranking. In our case:

- one pass retrieves candidates using dense vectors.
- one pass retrieves candidates using sparse vectors.

Each pass returns its own top results, which are then merged by the main query using Reciprocal Rank Fusion (RRF).

```python
client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=dense_vec,  using="dense",  limit=20),
        Prefetch(query=sparse_vec, using="sparse", limit=20),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
)
```

### Reciprocal Rank Fusion (RRF)

RRF combines results from multiple retrievers using rank positions only, rather than raw scores. This matters because dense and sparse scores live in completely different ranges and are not directly comparable.

The score of a document is computed as:

$$
\text{score}(d \in D) = \sum_{r_d \in R(d)} \frac{1}{k + \frac{r_d + 1}{w_r} - 1}
$$

Where:
- $D$ is the set of all documents across all results
- $R(d)$ is the set of rankings in which document $d$ appears
- $r_d$ is the rank of document $d$ in ranking $r$
- $k$ is a smoothing constant (default: 2, a smaller $k$ emphasizes top-ranked documents, while a larger $k$ smooths differences across ranks.)
- $w_r$ is the weight of ranking $r$ (default: 1 &rarr; equal contributions from sparse and dense)

A document that ranks well in both dense and sparse retrieval accumulates a higher total score and is ranked higher in the final list, even if it is not the top result in either list individually.

In [ ]:
def encode_query(qtext):
    dense_vec  = list(dense_model.query_embed([qtext]))[0].tolist()
    sparse_emb = list(sparse_model.query_embed([qtext]))[0]
    sparse_vec = SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist(),
    )
    return dense_vec, sparse_vec


def search_sparse_only(qtext, top_k=5):
    _, sparse_vec = encode_query(qtext)
    return client.query_points(
        collection_name=COLLECTION_NAME,
        query=sparse_vec,
        using="sparse",
        limit=top_k,
        with_payload=True,
    ).points


def search_dense_only(qtext, top_k=5):
    dense_vec, _ = encode_query(qtext)
    return client.query_points(
        collection_name=COLLECTION_NAME,
        query=dense_vec,
        using="dense",
        limit=top_k,
        with_payload=True,
    ).points


def search_hybrid_rrf(qtext, top_k=5, prefetch_k=30):
    dense_vec, sparse_vec = encode_query(qtext)
    # Prefetch candidates from both retrievers, then fuse with RRF
    return client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=dense_vec,  using="dense",  limit=prefetch_k),
            Prefetch(query=sparse_vec, using="sparse", limit=prefetch_k),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    ).points


CONFIGS = [
    ("sparse-only",        search_sparse_only),
    ("dense-only",         search_dense_only),
    ("sparse+dense (RRF)", search_hybrid_rrf),
]

eval_results = {}

for config_name, search_fn in CONFIGS:
    print(f"\nEvaluating: {config_name}")
    results = {}

    for qid, qtext in tqdm(eval_queries.items(), desc=f"  [{config_name}]"):
        hits = search_fn(qtext, top_k=5)
        results[qid] = {hit.payload["doc_id"]: hit.score for hit in hits}

    metrics = evaluate(
        qrels_ranx,
        Run(results),
        ["ndcg@5", "mrr", "precision@5", "recall@5"]
    )
    eval_results[config_name] = {
        "ndcg@5":      metrics["ndcg@5"],
        "mrr":         metrics["mrr"],
        "precision@5": metrics["precision@5"],
        "recall@5":    metrics["recall@5"],
    }
    r = eval_results[config_name]
    print(f"  NDCG@5={r['ndcg@5']:.4f}  MRR={r['mrr']:.4f}  "
          f"Precision@5={r['precision@5']:.4f}  Recall@5={r['recall@5']:.4f}")


Evaluating: sparse-only


  [sparse-only]: 100%|██████████| 300/300 [01:04<00:00,  4.66it/s]
/usr/local/lib/python3.12/dist-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


  NDCG@5=0.7779  MRR=0.7688  Precision@5=0.1853  Recall@5=0.8316

Evaluating: dense-only


  [dense-only]: 100%|██████████| 300/300 [01:05<00:00,  4.57it/s]


  NDCG@5=0.9028  MRR=0.8886  Precision@5=0.2147  Recall@5=0.9503

Evaluating: sparse+dense (RRF)


  [sparse+dense (RRF)]: 100%|██████████| 300/300 [01:05<00:00,  4.58it/s]

  NDCG@5=0.8944  MRR=0.8799  Precision@5=0.2140  Recall@5=0.9546


Dense-only retrieval is the clear winner. Sparse-only performs significantly worse than dense, which is expected given the nature of the data.

The hybrid configuration with equal RRF weights slightly underperforms dense-only on NDCG@5 and MRR, while marginally improving Recall@5. This is a textbook case of a weaker signal diluting a stronger one: by giving equal weight to sparse and dense, we allow BM25's noisier rankings to pull down results that dense retrieval was already ranking correctly.

This motivates a natural next step: **RRF weight tuning**.

---
## 6. RRF weight tuning

RRF supports per-prefetch weights, which control how much each retriever contributes to the final ranking. Since dense is clearly the stronger signal here, we will tweak the weights to give it more importance and see whether that outperforms the dense-only baseline.

We run the hybrid pipeline with different weight combinations, favoring dense over sparse, and compare against the previous results.

In [ ]:
rrf_weights = [
    (2.0, 1.0),   # dense mildly favored
    (3.0, 1.0),   # dense moderately favored
    (5.0, 1.0),   # dense strongly favored
    (10.0, 1.0),  # dense very strongly favored
]

for dense_w, sparse_w in rrf_weights:
    config_name = f"rrf (dense={dense_w}, sparse={sparse_w})"
    print(f"\nEvaluating: {config_name}")
    results = {}

    for qid, qtext in tqdm(eval_queries.items(), desc=f"  [{config_name}]"):
        dense_vec, sparse_vec = encode_query(qtext)
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=[
                Prefetch(query=dense_vec,  using="dense",  limit=20),
                Prefetch(query=sparse_vec, using="sparse", limit=20),
            ],
            query=RrfQuery(rrf=Rrf(weights=[dense_w, sparse_w])),
            limit=5,
            with_payload=True,
        ).points
        results[qid] = {hit.payload["doc_id"]: hit.score for hit in hits}

    metrics = evaluate(
        qrels_ranx,
        Run(results),
        ["ndcg@5", "mrr", "precision@5", "recall@5"]
    )
    eval_results[config_name] = {
        "ndcg@5":      metrics["ndcg@5"],
        "mrr":         metrics["mrr"],
        "precision@5": metrics["precision@5"],
        "recall@5":    metrics["recall@5"],
    }
    r = eval_results[config_name]
    print(f"  NDCG@5={r['ndcg@5']:.4f}  MRR={r['mrr']:.4f}  "
          f"Precision@5={r['precision@5']:.4f}  Recall@5={r['recall@5']:.4f}")


Evaluating: rrf (dense=2.0, sparse=1.0)


  [rrf (dense=2.0, sparse=1.0)]: 100%|██████████| 300/300 [01:04<00:00,  4.62it/s]


  NDCG@5=0.9066  MRR=0.8926  Precision@5=0.2167  Recall@5=0.9630

Evaluating: rrf (dense=3.0, sparse=1.0)


  [rrf (dense=3.0, sparse=1.0)]: 100%|██████████| 300/300 [01:04<00:00,  4.63it/s]


  NDCG@5=0.9049  MRR=0.8894  Precision@5=0.2167  Recall@5=0.9630

Evaluating: rrf (dense=5.0, sparse=1.0)


  [rrf (dense=5.0, sparse=1.0)]: 100%|██████████| 300/300 [01:05<00:00,  4.60it/s]


  NDCG@5=0.9039  MRR=0.8903  Precision@5=0.2160  Recall@5=0.9597

Evaluating: rrf (dense=10.0, sparse=1.0)


  [rrf (dense=10.0, sparse=1.0)]: 100%|██████████| 300/300 [01:05<00:00,  4.58it/s]

  NDCG@5=0.8979  MRR=0.8813  Precision@5=0.2160  Recall@5=0.9597


---
## 7. Results

In [ ]:
rows = []
for config_name, r in eval_results.items():
    rows.append({
        "Configuration":  config_name,
        "NDCG@5":         round(r["ndcg@5"], 4),
        "MRR":            round(r["mrr"], 4),
        "Precision@5":    round(r["precision@5"], 4),
        "Recall@5":       round(r["recall@5"], 4),
    })

rows.sort(key=lambda x: x["NDCG@5"], reverse=True)

header = list(rows[0].keys())
col_w  = {h: max(len(h), max(len(str(r[h])) for r in rows)) for h in header}
fmt    = "  ".join(f"{{:<{col_w[h]}}}" for h in header)
print(fmt.format(*header))
print("  ".join("-" * col_w[h] for h in header))
for row in rows:
    print(fmt.format(*[str(row[h]) for h in header]))

Configuration                 NDCG@5  MRR     Precision@5  Recall@5
----------------------------  ------  ------  -----------  --------
rrf (dense=2.0, sparse=1.0)   0.9066  0.8926  0.2167       0.963   
rrf (dense=3.0, sparse=1.0)   0.9049  0.8894  0.2167       0.963   
rrf (dense=5.0, sparse=1.0)   0.9039  0.8903  0.216        0.9597  
dense-only                    0.9028  0.8886  0.2147       0.9503  
rrf (dense=10.0, sparse=1.0)  0.8979  0.8813  0.216        0.9597  
sparse+dense (RRF)            0.8944  0.8799  0.214        0.9546  
sparse-only                   0.7779  0.7688  0.1853       0.8316  


Weighted RRF with dense=3.0 edges out dense-only, but the gains are marginal. Over-weighting dense at 10x starts to hurt, confirming there is a sweet spot where sparse still contributes just enough signal. Equal-weight RRF underperforms dense-only: a reminder that blindly combining retrievers without tuning can cost you.



---
## 8. Takeaways and closing thoughts

- **Why hybrid search didn't add dramatic value here:** the dense model was already performing very well on its own. On a larger corpus, or one with more technical jargon and keyword-heavy content, BM25 would have more opportunities to complement dense retrieval.

- **When hybrid search has real benefits:** corpora with specific identifiers, product codes, legal references, or precise scientific terminology where exact term matching matters and semantic similarity alone is not enough.

- **For more complex use cases:** you may also want to inject business logic through **score boosting**. For example, favouring research papers that are more recent or more highly cited. You can also add a **late interaction reranking** step with models like ColBERT, which captures token-level precision between query and document. Given its higher latency and memory consumption, it is applied only to a small pool of candidates retrieved by earlier prefetch stages.

- **Alternative fusion strategies:** RRF is not the only fusion method available in Qdrant. You can also use Distribution-Based Score Fusion (DBSF), which normalizes scores across retrievers before combining them.

- Read more about [Hybrid and Multi-stage Queries](https://qdrant.tech/documentation/search/hybrid-queries) in the Qdrant documentation.
